# LoRA Fine-Tuning & Few-Shot Prompting on GSM8K

### Project Roadmap

* Task 1: Benchmark the base model to establish the performance floor.
* Task 2: Apply LoRA fine-tuning with GSM8K training data and study scaling behavior.
* Task 3: Investigate few-shot prompting and its interaction with fine-tuning.
* Task 4: Discover that data quality matters more than data quantity, and push accuracy further.

### Evaluation Protocol
* Test set: 100 questions from the GSM8K test split (fixed across all experiments).
* Metric: Accuracy = correct answers / total questions

# Task 1 — Baseline: Zero-Shot Performance

Before any fine-tuning, we measure how well the base Qwen2.5-1.5B-Instruct model solves GSM8K problems out of the box. This establishes a performance floor that all subsequent interventions (fine-tuning, prompting) will be compared against.

In [ ]:
!pip install trl peft accelerate -q

In [ ]:
import os
import re
import json
import math
import gc
from datetime import datetime

import matplotlib.pyplot as plt
import torch
from tqdm import tqdm
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from trl import SFTTrainer, SFTConfig

HF_TOKEN = os.environ.get('HF_TOKEN')
MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU detected - running on CPU')

In [ ]:
SYSTEM_PROMPT = (
    "Answer the question step-by-step. Your final answer must be a number enclosed in \\boxed{}."
)

In [ ]:
# ── Answer extraction ──

def extract_boxed(text):
    """Extract content from the last \boxed{}, handling nested braces and variations like 'oxed{}'."""
    # Find all occurrences of \boxed{ or oxed{
    matches = list(re.finditer(r"\\?boxed{", text))
    if not matches:
        return None

    # Take the last match
    last_match = matches[-1]
    start_content = last_match.end() # Position right after the opening brace

    brace_level = 0
    for i in range(start_content, len(text)):
        if text[i] == '{':
            brace_level += 1
        elif text[i] == '}':
            if brace_level == 0: # Found the matching closing brace for the last \boxed
                return text[start_content:i]
            brace_level -= 1
    return None


def extract_ground_truth(raw_answer, gt_format):
    """Extract the ground-truth answer string from the dataset's answer field."""
    if gt_format == "hashmarks":
        m = re.search(r"####\s*(.+)", raw_answer)
        if m:
            return m.group(1).strip().replace(",", "")
        return raw_answer.strip()
    if gt_format == "boxed":
        boxed = extract_boxed(raw_answer)
        if boxed is not None:
            return boxed.strip()
        return raw_answer.strip()
    return raw_answer.strip()


def extract_model_answer(text):
    """Extract the final numerical answer from the model's output using \boxed{}."""
    boxed = extract_boxed(text)
    if boxed is not None:
        return boxed.strip().replace(",", "")
    return None




# ── Model loading ──

def load_model(model_name=MODEL_NAME, lora_path=None):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="eager",
    )

    if lora_path:
        print(f"Loading LoRA adapter from {lora_path}")
        model = PeftModel.from_pretrained(model, lora_path)

    model.eval()
    tag = f" + LoRA({lora_path})" if lora_path else " (base)"
    print(f"Loaded: {model_name}{tag}")
    return model, tokenizer


def cleanup(*objects):
    """Delete model/tokenizer objects and free GPU memory."""
    for obj in objects:
        del obj
    gc.collect()
    torch.cuda.empty_cache()
    print(f"GPU memory freed. Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")


# ── Prompt building & batched generation ──

def build_prompts(tokenizer, questions):
    """Build chat-formatted prompt strings for a list of questions."""
    prompts = []
    for q in questions:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": q},
        ]
        prompts.append(
            tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
        )
    return prompts


def generate_batch(model, tokenizer, questions):
    """Generate responses for a batch of questions in one forward pass."""
    prompts = build_prompts(tokenizer, questions)
    inputs = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True
    ).to(model.device)
    prompt_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=2048, do_sample=False)

    responses = []
    for i in range(len(questions)):
        new_tokens = out[i][prompt_len:]
        responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True))
    return responses


# ── Evaluation runner ──

def evaluate_gsm8k(model, tokenizer, num_samples=100, batch_size=16):
    """Zero-shot eval on GSM8K test set. Returns accuracy and records."""
    dataset = load_dataset("gsm8k", "main", split="test")

    # Select a subset of the dataset
    if num_samples > 0:
        dataset = dataset.select(range(num_samples))

    questions = [entry["question"] for entry in dataset]
    ground_truths = [extract_ground_truth(entry["answer"], gt_format="hashmarks") for entry in dataset]

    records = []
    correct_count = 0

    # Initialize tqdm with total number of questions for per-sample updates
    pbar = tqdm(total=len(questions), desc="Evaluating GSM8K", unit="sample")

    for i in range(0, len(questions), batch_size):
        batch_questions = questions[i : i + batch_size]
        batch_ground_truths = ground_truths[i : i + batch_size]

        responses = generate_batch(model, tokenizer, batch_questions)

        for j, response in enumerate(responses):
            model_answer = extract_model_answer(response)
            gt_answer = batch_ground_truths[j]
            question = batch_questions[j]

            is_correct = False
            if model_answer is not None and gt_answer is not None:
                try:
                    model_val = float(model_answer)
                    gt_val = float(gt_answer)
                    is_correct = math.isclose(model_val, gt_val, rel_tol=1e-5, abs_tol=1e-5)
                except ValueError:
                    is_correct = (model_answer.strip() == gt_answer.strip())

            if is_correct:
                correct_count += 1

            records.append({
                "question": question,
                "ground_truth": gt_answer,
                "model_response": response,
                "model_answer": model_answer,
                "is_correct": is_correct,
            })
            pbar.update(1) # Update progress bar for each sample
    pbar.close()

    acc = correct_count / len(questions) if len(questions) > 0 else 0.0
    return acc, records


In [ ]:
model, tokenizer = load_model()
base_acc, base_records = evaluate_gsm8k(model, tokenizer, num_samples=100) 
print(f"\nBase model zero-shot accuracy: {base_acc:.2%}")
none_records = [record for record in base_records if record['model_answer'] == None]

if len(base_records) > 0:
    none_percentage = (len(none_records) / len(base_records)) * 100
    print(f"Percentage of answers where extraction failed (model_answer is None): {none_percentage:.2f}%")
else:
    print("No records found to evaluate.")

# Display some records to check extraction
print("\n--- Sample Records (first 3) ---")
for i, record in enumerate(base_records[:5]):
    print(f"Question: {record['question'][:100]}...")
    print(f"Ground Truth: {record['ground_truth']}")
    print(f"Model Answer: {record['model_answer']}")
    print(f"Raw Model Response: {record['model_response']}...") # Display first 200 chars of raw response
    print(f"Correct: {record['is_correct']}")
    print("------------------------------")

cleanup(model, tokenizer)

In [ ]:
if none_records:
    print("\n--- Records where model_answer is None ---")
    for record in none_records:
        print(f"Question: {record['question'][:100]}...")
        print(f"Raw Model Response: {record['model_response'][-100:]}...")
        print(f"Model Answer: {record['model_answer']}")
        print("------------------------------")
else:
    print("\nNo records found where model_answer is None.")

In [ ]:
incorrect_records = [record for record in base_records if not record['is_correct']]

print("--- Incorrect Records (first 3) ---")
for i, record in enumerate(incorrect_records[:3]):
    print(f"Question: {record['question']}...")
    print(f"Ground Truth: {record['ground_truth']}")
    print(f"Model Answer: {record['model_answer']}")
    print(f"Raw Model Response: {record['model_response'][:200]}...")
    print(f"Correct: {record['is_correct']}")
    print("------------------------------")

1. Model added house price and repair cost to attain cost of house, which was incorrect. It also used .15 instead of 2.5 (since it is increased by 150% it shoudl be 1 + 1.5) I would classify this as a failure to ground the mathematical operations in the narrative context.

2. Model assumed chickens were supposed to receive 10 cups. However, the problem explicity stated 3 cups. I would classify this as information retrieval failure/logical hallucination

3. The model mistinterpreted the discount as a geometric discount, assuming the discount applied for subsequent purchases of the glass and not every second glass. I would also classify this as a failure to ground the mathematical operations in the narrative context.

It seems like the model struggles with truly understanding the problems within their narrative context. The model also seems to make mistakes with simpole mathematical errors (i.e. multiplying .15 instead of 2.5 for increase in 150%)

# Task 2 — LoRA Fine-Tuning on GSM8K

The base model struggles with math. We apply LoRA SFT using the GSM8K training split, starting small and scaling up to study how data quantity affects performance.

The training pipeline has three key ingredients:
1. **Data formatting**: Each GSM8K training example is converted into a chat-style interaction (system prompt + user question + assistant solution).
2. **LoRA configuration**: Adapters are inserted into selected attention projection layers. Base weights remain frozen; only adapter parameters are trained.
3. **Completion-only loss**: Loss is computed only on the assistant's response tokens.

Default hyperparameters:

| Hyperparameter | Default | Notes |
|---|---|---|
| LoRA rank (r) | 8 | Controls adapter capacity |
| LoRA alpha | 16 | Scaling factor (2r) |
| LoRA dropout | 0.05 | |
| Learning rate | 2e-4 | Cosine schedule with 5% warmup |
| Epochs | 1 | |
| Per-device batch size | 8 | |
| Gradient accumulation | 4 | Effective batch size = 32 |
| Max sequence length | 1024 | |
| Target modules | Attention only | q, k, v, o proj |

LoRA rank:
* controls the dimensionality of the low-rank adapter matrices
* increasing it increase VRAM usage and risks overfitting, but the model will be able to learn more complex patterns and reasoning
* decreasing it will make the model much more memory efficient but the model may not be as robust and is subject to underfitting


LoRA alpha:
* scaling factor of the weights. Determines how much weight the newly learned parameters have in relation to the model original pre-trained knowledge
* Increasing it helps the model learn faster and the updates become more influential, however this can cause the model's logic to break
* Decreasing it causes the updates to be more incremental/subtle, but this may lead to it not adapting to the new reasoning at all.


Gradient accumulation:
* The number of steps the model processes before it actually updates its internal weights
* increasing it essentially creates a larger "batch" size, which leads to smoother and more stable gradients. However, this makes the training time significantly longer
* decreasing it makes the model update its weights more frequently, which makes the gradients more noisy and more inconsistent.


In [ ]:
# ── Load model for training ──
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "right"  # right padding for training
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)

# ── LoRA config (attention-only, rank 8) ──
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

(a) 1,545,893,376

(b) 2,179,072

(c) 0.1410%

The percentage is small as LoRA signficantly reduces the # of trainable parameters by essentially freezing the majority of the pre-trained model weights. Instead of fine-tuning all of the parameters, LoRA injects low-rank matrices/adapters into specific layers of the model, and only the paramters of these small adapter matrices are trained, which constitutes only 0.1410% of the toal parameters. This saves a significant amount of computational resources while still enabling the model to adapt to new tasks.

## Task 2.2 — Training with Increasing Data

The GSM8K training split contains 7,473 question-answer pairs. We start with 1,000 examples to build intuition and validate the setup, then scale to 3,000 to measure the effect of additional data.

In [ ]:
# ── Load model for training ──
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "right"  # right padding for training
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)

# ── LoRA config (attention-only, rank 8) ──
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ── Load & format GSM8K training data (1k) ──
GSM8K_TRAIN_SAMPLES = 1000



dataset = load_dataset("gsm8k", "main", split="train")
dataset = dataset.select(range(GSM8K_TRAIN_SAMPLES))

# Apply formatting
train_data = dataset.map(
    format_example,
    remove_columns=dataset.column_names
)

# Quick sanity check
print("Example formatted training sample:\n")
print(train_data[0]["text"])

# ── Training config ──
SFT_1K_DIR = "1kpath"

training_args = SFTConfig(
    output_dir=SFT_1K_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    max_length=1024,
    completion_only_loss=True,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data, # your training data
    processing_class=tokenizer,
)

print(f"\nStarting SFT {GSM8K_TRAIN_SAMPLES} training...\n")
trainer.train()

# ── Save adapter ──
adapter_path = os.path.join(SFT_1K_DIR, "final_adapter")
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"\nAdapter saved to {adapter_path}")

cleanup(model, tokenizer, trainer)


In [ ]:
model, tokenizer = load_model(lora_path="1kpath/final_adapter")
sft1k_acc, sft1k_records = evaluate_gsm8k(model, tokenizer, num_samples=100)
print(f"\nSFT 1k zero-shot accuracy: {sft1k_acc:.2%}")
cleanup(model, tokenizer)

In [ ]:
none_sft_records = [record for record in sft1k_records if record['model_answer'] is None]

if none_sft_records:
    print(f"\n--- Sample records ({len(none_sft_records)} total) where model_answer is None after SFT ---")
    for i, record in enumerate(none_sft_records[:5]): # Display first 5
        print(f"Question: {record['question']}")
        print(f"Ground Truth: {record['ground_truth']}")
        print(f"Model Response: {record['model_response']}...")
        print(f"Model Answer (extracted): {record['model_answer']}")
        print("--------------------------------------------------")
else:
    print("No records found where model_answer is None after SFT.")

I think that scaling from 1000 to 3000 is worth the additional compute. Right now the jump from baseline to 1000 increased it by 2%, which is marginal but still signficant enough improvement. I think the gain will be another 2% potentially. I will just go to 3000 and not all, as I think the compute time of all examples would not be worthe marginal performance boost.

In [ ]:
# ── Load model for training ──
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "right"  # right padding for training
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)

# ── LoRA config (attention-only, rank 8) ──
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ── Load & format GSM8K training data (3k) ──
GSM8K_TRAIN_SAMPLES = 3000



dataset = load_dataset("gsm8k", "main", split="train")
dataset = dataset.select(range(GSM8K_TRAIN_SAMPLES))

# Apply formatting
train_data = dataset.map(
    format_example,
    remove_columns=dataset.column_names
)

# Quick sanity check
print("Example formatted training sample:\n")
print(train_data[0]["text"])

# ── Training config ──
SFT_3K_DIR = "3kpath"

training_args = SFTConfig(
    output_dir=SFT_3K_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    max_length=1024,
    completion_only_loss=True,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data, # your training data
    processing_class=tokenizer,
)

print(f"\nStarting SFT {GSM8K_TRAIN_SAMPLES} training...\n")
trainer.train()

# ── Save adapter ──
adapter_path = os.path.join(SFT_3K_DIR, "final_adapter")
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"\nAdapter saved to {adapter_path}")

cleanup(model, tokenizer, trainer)


In [ ]:
model, tokenizer = load_model(lora_path="3kpath/final_adapter")
sft3k_acc, sft3k_records = evaluate_gsm8k(model, tokenizer, num_samples=100)
print(f"\nSFT 3k zero-shot accuracy: {sft3k_acc:.2%}")
cleanup(model, tokenizer)

In [ ]:
training_examples = [0, 1000, 3000]
accuracies = [base_acc, sft1k_acc, sft3k_acc]

plt.figure(figsize=(8, 6))
plt.plot(training_examples, accuracies, marker='o', linestyle='-', color='b')

plt.xlabel('Number of Training Examples')
plt.ylabel('Accuracy')
plt.title('Accuracy vs. Number of Training Examples')

plt.grid(True)

plt.show()

Accuracies observed:
*   Base model (0 examples): 40.00%
*   SFT with 1,000 examples: 42.00%
*   SFT with 3,000 examples: 43.00%

The plot shows an increasing trend in accuracy as the number of training examples increases. There was a 2% jump from the baseline to 1,000 examples, and then a smaller 1% jump from 1,000 to 3,000 examples. This trend suggests diminishing returns in data scaling for SFT. While adding more data initially provides significant gains, the improvement lessens as the dataset grows larger. This indicates that at some point, adding more data may not yield proportional increases in performance, which validates my idea that scaling up to all examples would not be worth the computational resources.

## Task 2.3 — Qualitative Analysis

Comparison of the base model and best SFT model on failure examples identified in Task 1.

In [ ]:
sft_model, sft_tokenizer = load_model(lora_path="3kpath/final_adapter")
print(f"Loaded best SFT model with path: {'3kpath/final_adapter'}")

In [ ]:
print("\n--- Comparison of Base vs. SFT Model on Failure Examples ---")

for i, base_incorrect_record in enumerate(incorrect_records[:3]):
    question = base_incorrect_record['question']
    ground_truth = base_incorrect_record['ground_truth']

    # Find the corresponding record in sft3k_records using the question
    sft_corresponding_record = next((rec for rec in sft3k_records if rec['question'] == question), None)

    print(f"\nExample {i + 1}:")
    print(f"Question: {question}")
    print(f"Ground Truth: {ground_truth}")
    print("\n--- Base Model Response ---")
    print(f"Model Answer: {base_incorrect_record['model_answer']}")
    print(f"Raw Response (excerpt): {base_incorrect_record['model_response'][:300]}...") # Truncate for brevity
    print(f"Correct: {base_incorrect_record['is_correct']}")

    print("\n--- SFT Model Response ---")
    if sft_corresponding_record:
        print(f"Model Answer: {sft_corresponding_record['model_answer']}")
        print(f"Raw Response (excerpt): {sft_corresponding_record['model_response'][:300]}...") # Truncate for brevity
        print(f"Correct: {sft_corresponding_record['is_correct']}")
    else:
        print("Corresponding SFT record not found.")

    print("--------------------------------------------------")


Example 1
* The SFT model did not fix this error. While the numerical answer changed, the underlying logical misinterpretation of the percentage increase and the calculation of the final value persisted. The SFT model's reasoning still shows a misunderstanding of how the 150% increase should be applied and performs incorrect arithmetic (0.15 instead of 1.5 or 2.5 for 150% increase). Neither model correctly understood that a 150% increase means the new value is 2.5 times the base value.

Example 2
* The SFT model did not fix the core reasoning error. While it produced a boxed answer, it introduced new reasoning flaws and significantly deviated from the problem's explicit conditions, leading to an even more incorrect answer. The model still fails to correctly use the "3 cups per chicken per day" information.

Example 3
* The SFT model did not fix this error. In fact, it introduced a more fundamental arithmetic error in calculating 40% of 200, leading to a different incorrect answer. The core issue of correctly tracking the progress, restart, and total time calculation remains unsolved.

Based on these three failure examples, the SFT model did not successfully fix the errors. In some cases, it introduced new errors or presented different incorrect reasoning. The primary failure modes observed in both models are a combination of reasoning/logical errors, arithmetic mistakes, and information retrieval failure

While the overall accuracy saw a slight increase, these specific complex reasoning problems highlight that SFT with 3,000 examples was not sufficient to overcome these deeper logical and comprehension challenges

See above. The SFT model still failed with the same questions as the base model.

# Task 3 — Few-Shot Prompting

Fine-tuning changes model weights. An orthogonal approach is to change the prompt: including a small number of worked examples before the test question steers the model toward a desired reasoning style and consistent output format, without updating any parameters.

In k-shot prompting, k complete (question, solution) pairs are prepended as user/assistant turns before the actual test question. A fixed pool of demonstration examples is used across all models evaluated to isolate the effect of the model from the effect of the examples.

## Task 3.2 — Few-Shot Experiments

Evaluate k=3 shot prompting on the base model and the best SFT model, comparing against the zero-shot baselines.

In [ ]:
dataset = load_dataset("gsm8k", "main", split="train")

demo_dataset = dataset.select(range(3))

demonstration_examples = []

for example in demo_dataset:
    question = example["question"].strip()
    answer = clean_answer(example["answer"])

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ]

    formatted_demo = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    demonstration_examples.append(formatted_demo)

print("--- Formatted Demonstration Examples (k=3) ---")
for i, demo in enumerate(demonstration_examples):
    print(f"\nExample {i+1}:\n{demo}")
    print("---------------------------------------------")

In [ ]:
def build_prompts_k_shot(tokenizer, questions, demonstration_examples):
    """Build chat-formatted prompt strings for a list of questions, including k-shot examples."""
    prompts = []
    for q in questions:
        full_prompt = "".join(demonstration_examples)

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": q},
        ]
        full_prompt += tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        prompts.append(full_prompt)
    return prompts

def generate_batch_k_shot(model, tokenizer, questions, demonstration_examples):
    """Generate responses for a batch of questions with k-shot prompting."""
    prompts = build_prompts_k_shot(tokenizer, questions, demonstration_examples)
    inputs = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True
    ).to(model.device)
    prompt_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=2048, do_sample=False)

    responses = []
    for i in range(len(questions)):
        new_tokens = out[i][prompt_len:]
        responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True))
    return responses

def evaluate_gsm8k_k_shot(model, tokenizer, demonstration_examples, num_samples=100, batch_size=16):
    """Zero-shot eval on GSM8K test set with k-shot prompting. Returns accuracy and records."""
    dataset = load_dataset("gsm8k", "main", split="test")

    if num_samples > 0:
        dataset = dataset.select(range(num_samples))

    questions = [entry["question"] for entry in dataset]
    ground_truths = [extract_ground_truth(entry["answer"], gt_format="hashmarks") for entry in dataset]

    records = []
    correct_count = 0

    pbar = tqdm(total=len(questions), desc="Evaluating GSM8K with k-shot", unit="sample")

    for i in range(0, len(questions), batch_size):
        batch_questions = questions[i : i + batch_size]
        batch_ground_truths = ground_truths[i : i + batch_size]

        responses = generate_batch_k_shot(model, tokenizer, batch_questions, demonstration_examples)

        for j, response in enumerate(responses):
            model_answer = extract_model_answer(response)
            gt_answer = batch_ground_truths[j]
            question = batch_questions[j]

            is_correct = False
            if model_answer is not None and gt_answer is not None:
                try:
                    model_val = float(model_answer)
                    gt_val = float(gt_answer)
                    is_correct = math.isclose(model_val, gt_val, rel_tol=1e-5, abs_tol=1e-5)
                except ValueError:
                    is_correct = (model_answer.strip() == gt_answer.strip())

            if is_correct:
                correct_count += 1

            records.append({
                "question": question,
                "ground_truth": gt_answer,
                "model_response": response,
                "model_answer": model_answer,
                "is_correct": is_correct,
            })
            pbar.update(1)
    pbar.close()

    acc = correct_count / len(questions) if len(questions) > 0 else 0.0
    return acc, records

print("\n--- Evaluating Base Model with 3-shot Prompting ---")
base_model_kshot, base_tokenizer_kshot = load_model()
base_kshot_acc, base_kshot_records = evaluate_gsm8k_k_shot(
    base_model_kshot, base_tokenizer_kshot, demonstration_examples, num_samples=100
)
print(f"\nBase model 3-shot accuracy: {base_kshot_acc:.2%}")
cleanup(base_model_kshot, base_tokenizer_kshot)

In [ ]:
print(
    "\n--- Evaluating SFT Model (3k examples) with 3-shot Prompting ---"
)
sft_model_kshot, sft_tokenizer_kshot = load_model(lora_path="3kpath/final_adapter")
sft_kshot_acc, sft_kshot_records = evaluate_gsm8k_k_shot(
    sft_model_kshot, sft_tokenizer_kshot, demonstration_examples, num_samples=100
)
print(f"\nSFT model (3k) 3-shot accuracy: {sft_kshot_acc:.2%}")
cleanup(sft_model_kshot, sft_tokenizer_kshot)

In [ ]:
print("\n--- K-shot Prompting Results (k=3) ---")
print(f"Baseline (0-shot): {base_acc:.2%}")
print(f"Baseline (3-shot): {base_kshot_acc:.2%}")
print(f"Improvement (Baseline): {base_kshot_acc - base_acc:.2%}\n")

print(f"SFT Model (3k examples, 0-shot): {sft3k_acc:.2%}")
print(f"SFT Model (3k examples, 3-shot): {sft_kshot_acc:.2%}")
print(f"Improvement (SFT 3k): {sft_kshot_acc - sft3k_acc:.2%}")


Few-shot prompting does not seem to help with the base model. If fact, it significantly decreased accuracy. This is likely due to the examples "distracting" the model, making it focus more on mimicing the examples rather than learning the logic behind the problem.  I observed this personally when trying to test prompt the original base model as well.

Since the SFT 3k model has already been trained on the specific question/step-by-step/answer structure, few-shot examples act as a reinforcement signal that stabilizes its existing knowledge. Because the model is already "aligned" to the task, the demonstrations provide helpful templates for difficult logic rather than confusing noise, leading to a 5% improvement.


# Task 4 — Quality over Quantity

Scaling (more training examples, longer prompts, more shots) helps, but gains slow down. The bottleneck becomes data quality rather than quantity. Training on fewer but higher-quality, well-structured solutions can lead to larger improvements than adding more noisy data.

In [ ]:
none_sft3k_count = sum(1 for record in sft3k_records if record['model_answer'] is None)
none_base_kshot_count = sum(1 for record in base_kshot_records if record['model_answer'] is None)
none_sft_kshot_count = sum(1 for record in sft_kshot_records if record['model_answer'] is None)

print(f"Number of 'None' model_answers in SFT 3k records: {none_sft3k_count}")
print(f"Number of 'None' model_answers in Base K-shot records: {none_base_kshot_count}")
print(f"Number of 'None' model_answers in SFT K-shot records: {none_sft_kshot_count}")

* arithmetic reliability

The model fails at basic math in some cases. An example is the model formatting 150% as .15
* multi-step planning / long-horizon reasoning

from what I could tell, the model didn't have too much issues with this. The model was able to understand and plan what the end goal is and the steps needed to get there, but fails in the beginning due to errors like problem comprehension and arithmetic issues.
* problem comprehension

From what I have observed, I believe the biggest performance limiting factor is problem comprehension. As gone over in the previous examples, the model simply fails to comprehend the context and situation that the math problem is presenting. It misses simple details such as the cups of chicken feed or the idea of profit in the context of house flipping. This fundamentally leads to the model being lead in the completely wrong direction, and then the model makes simple arithmetic mistakes after.

* output consistency / extraction failures

 The next biggest issue seems to be the output consistency and extraction failures. The base model had a 9% output format failure, and the base k-shot one had even worse. The SFT model had no issues, but had a 2% output format failure rate with the k-shot prompting.

* training data quality (solution style, structure, or noise).

Training data quality may also be a limiting factor, but it is hard to say for sure unless we do extensive testing with better quality training data. From what I observe of the solutions provided, it does not provide the model with the necessary understanding of the context and the step-by-step reasoning needed for the model to perform properly.



# Task 5 — Open Challenge: Push Toward the Ceiling

Goal: beat the best score from Tasks 1-3 by proposing a targeted hypothesis, running controlled experiments, and analyzing what worked and why.

Approach: increase LoRA rank (r=8 to r=16 and r=32) to allow the model to learn more complex mathematical reasoning patterns.

(a) Hypothesis: As observed in previous examples, the model consistently struggles at problem comprehension. Increasing the LoRA rank from the current r = 8 will allow the model to learn more complex and nuanced mathematical and reasoning patterns and potentially lead to better accuracy

(b) We will only tune the LoRA rank, starting with the baseline of 8. Due to time and resource constraints, I will only test with r=16 and the move to r=32.

In [ ]:
# ── Load model for training ──
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "right"  # right padding for training
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)

# ── LoRA config (attention-only, rank 16) ──
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16, # Increased LoRA rank
    lora_alpha=32, # Typically 2*r
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ── Load & format GSM8K training data (3k) ──
GSM8K_TRAIN_SAMPLES_R16 = 3000 # Using a new variable name for clarity

dataset = load_dataset("gsm8k", "main", split="train")
dataset = dataset.select(range(GSM8K_TRAIN_SAMPLES_R16))

# Apply formatting
train_data_r16 = dataset.map(
    format_example,
    remove_columns=dataset.column_names
)

# Quick sanity check
print("\nExample formatted training sample (r=16):\n")
print(train_data_r16[0]["text"])

# ── Training config ──
SFT_3K_R16_DIR = "3k_r16_path"

training_args_r16 = SFTConfig(
    output_dir=SFT_3K_R16_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    max_length=1024,
    completion_only_loss=True,
)

trainer_r16 = SFTTrainer(
    model=model,
    args=training_args_r16,
    train_dataset=train_data_r16,
    processing_class=tokenizer,
)

print(f"\nStarting SFT {GSM8K_TRAIN_SAMPLES_R16} training with r=16...\n")
trainer_r16.train()

# ── Save adapter ──
adapter_path_r16 = os.path.join(SFT_3K_R16_DIR, "final_adapter")
model.save_pretrained(adapter_path_r16)
tokenizer.save_pretrained(adapter_path_r16)
print(f"\nAdapter saved to {adapter_path_r16}")

cleanup(model, tokenizer, trainer_r16)


# ── Evaluation with 3-shot prompting ──
print("\n--- Evaluating SFT Model (3k examples, r=16) with 3-shot Prompting ---")
sft_r16_kshot_model, sft_r16_kshot_tokenizer = load_model(lora_path=adapter_path_r16)
sft_r16_kshot_acc, sft_r16_kshot_records = evaluate_gsm8k_k_shot(
    sft_r16_kshot_model, sft_r16_kshot_tokenizer, demonstration_examples, num_samples=100
)
print(f"\nSFT model (3k, r=16) 3-shot accuracy: {sft_r16_kshot_acc:.2%}")
cleanup(sft_r16_kshot_model, sft_r16_kshot_tokenizer)


In [ ]:
# ── Load model for training ──
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "right"  # right padding for training
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)

# ── LoRA config (attention-only, rank 32) ──
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=32, # Increased LoRA rank
    lora_alpha=64, # Typically 2*r
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ── Load & format GSM8K training data (3k) ──
GSM8K_TRAIN_SAMPLES_R32 = 3000

dataset = load_dataset("gsm8k", "main", split="train")
dataset = dataset.select(range(GSM8K_TRAIN_SAMPLES_R32))

# Apply formatting
train_data_r32 = dataset.map(
    format_example,
    remove_columns=dataset.column_names
)

# Quick sanity check
print("\nExample formatted training sample (r=16):\n")
print(train_data_r32[0]["text"])

# ── Training config ──
SFT_3K_R32_DIR = "3k_r32_path"

training_args_r32 = SFTConfig(
    output_dir=SFT_3K_R32_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    max_length=1024,
    completion_only_loss=True,
)

trainer_r32 = SFTTrainer(
    model=model,
    args=training_args_r32,
    train_dataset=train_data_r32,
    processing_class=tokenizer,
)

print(f"\nStarting SFT {GSM8K_TRAIN_SAMPLES_R32} training with r=32...\n")
trainer_r32.train()

# ── Save adapter ──
adapter_path_r32 = os.path.join(SFT_3K_R32_DIR, "final_adapter")
model.save_pretrained(adapter_path_r32)
tokenizer.save_pretrained(adapter_path_r32)
print(f"\nAdapter saved to {adapter_path_r32}")

cleanup(model, tokenizer, trainer_r32)


# ── Evaluation with 3-shot prompting ──
print("\n--- Evaluating SFT Model (3k examples, r=32) with 3-shot Prompting ---")
sft_r32_kshot_model, sft_r32_kshot_tokenizer = load_model(lora_path=adapter_path_r32)
sft_r32_kshot_acc, sft_r32_kshot_records = evaluate_gsm8k_k_shot(
    sft_r32_kshot_model, sft_r32_kshot_tokenizer, demonstration_examples, num_samples=100
)
print(f"\nSFT model (3k, r=32) 3-shot accuracy: {sft_r32_kshot_acc:.2%}")
cleanup(sft_r32_kshot_model, sft_r32_kshot_tokenizer)


In [ ]:
none_r16_kshot_count = sum(1 for record in sft_r16_kshot_records if record['model_answer'] is None)
none_r32_kshot_count = sum(1 for record in sft_r32_kshot_records if record['model_answer'] is None)

print(f"Number of 'None' model_answers in SFT 3k r=16 records: {none_r16_kshot_count}")
print(f"Number of 'None' model_answers in SFT 3k r=32 records: {none_r32_kshot_count}")

(c) Results:

In [ ]:
lora_ranks = [8, 16, 32]
accuracies = [sft_kshot_acc, sft_r16_kshot_acc, sft_r32_kshot_acc]

plt.figure(figsize=(8, 6))
plt.plot(lora_ranks, accuracies, marker='o', linestyle='-', color='g')

plt.xlabel('LoRA Rank (r)')
plt.ylabel('3-shot Accuracy')
plt.title('3-shot Accuracy vs. LoRA Rank (SFT 3k Examples)')

plt.xticks(lora_ranks) # Ensure x-axis ticks are at the exact rank values
plt.grid(True)

plt.show()

print(f"\nSummary of accuracies for 3k SFT with 3-shot prompting:\n")
print(f"  r=8:   {sft_kshot_acc:.2%}")
print(f"  r=16:  {sft_r16_kshot_acc:.2%}")
print(f"  r=32:  {sft_r32_kshot_acc:.2%}")

(d) Analysis:

From these results, we can observe a clear trend: increasing the LoRA rank generally leads to an improvement in the model's 3-shot accuracy. The jump from r=8 to r=16 yielded a significant 7.00% increase in accuracy (from 48% to 55%), which strongly supports my hypothesis that a higher LoRA rank could enable the model to learn more complex and nuanced mathematical and reasoning patterns. However, the jump from r=16 an r=32 only yielded a 1% increase (55% to 56%) which suggests that the performance will plateau past a certain point and would not be worth the training time and memory.

An unexpected result: Initially, I thought the rank would have a higher plateau, and that the only tradeoff with increasing rank was space/memory constrains. However, the small change from 16 to 32 of only 1% was pretty unexpected. This suggests to me that expanding LoRA parameters themselves can only go so far - methods like improving training data may need to be employed to push it further.